# 🧠 QwenRaggity V2: The Knowledge Engine
Bu notebook, **GraphRAG** ve **LLM Wiki** mimarilerini birleştiren, **LanceDB** ve **Qwen2.5-14B** ile güçlendirilmiş gelişmiş RAG sistemini kurar.

In [ ]:
# 1. Temizlik ve Kurulum
import os
os.system("pkill -f streamlit")
os.system("pkill -f pinggy")

!pip install -q lancedb tantivy==0.22.0 streamlit transformers accelerate bitsandbytes rank_bm25 scikit-learn numpy Pillow langchain-text-splitters sentence-transformers docling streamlit-pdf-viewer networkx requests
print("✅ Bağımlılıklar kuruldu!")

In [ ]:
%%writefile llm_utils.py
# -*- coding: utf-8 -*-
import os, torch, requests, json
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import CrossEncoder

class CustomEmbedder:
    def __init__(self, model_name="BAAI/bge-m3", device="cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
    def encode(self, sentences):
        if isinstance(sentences, str): sentences = [sentences]
        encoded_input = self.tokenizer(sentences, padding=True, truncation=True, max_length=512, return_tensors='pt').to(self.device)
        with torch.no_grad():
            model_output = self.model(**encoded_input)
        sentence_embs = model_output[0][:, 0]
        sentence_embs = F.normalize(sentence_embs, p=2, dim=1)
        return sentence_embs.cpu().numpy()

class RerankerEngine:
    def __init__(self, model_name='BAAI/bge-reranker-v2-m3', device="cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        self.model = CrossEncoder(model_name, max_length=512, device=self.device)
    def predict(self, pairs):
        return self.model.predict(pairs)

class LLMEngine:
    def __init__(self, model_name="qwen2.5:14b", url="http://localhost:11434/api/chat"):
        self.model_name = model_name
        self.url = url
    def generate(self, messages, temperature=0.4, stream=False):
        payload = {"model": self.model_name, "messages": messages, "stream": stream, "options": {"temperature": temperature, "top_p": 0.9}}
        if stream:
            return self._stream_response(payload)
        else:
            try:
                r = requests.post(self.url, json=payload, timeout=120)
                return r.json().get("message", {}).get("content", "")
            except: return "Ollama Baglanti Hatasi"
    def _stream_response(self, payload):
        with requests.post(self.url, json=payload, stream=True) as r:
            for line in r.iter_lines():
                if line:
                    chunk = json.loads(line)
                    yield chunk["message"]["content"]

_embedder = None; _reranker = None; _llm = None
def get_embedder():
    global _embedder
    if _embedder is None: _embedder = CustomEmbedder()
    return _embedder
def get_reranker():
    global _reranker
    if _reranker is None: _reranker = RerankerEngine()
    return _reranker
def get_llm():
    global _llm
    if _llm is None: _llm = LLMEngine()
    return _llm

In [ ]:
%%writefile ingestion_engine.py
# -*- coding: utf-8 -*-
import os, hashlib, json
from datetime import datetime
from docling.document_converter import DocumentConverter
from llm_utils import get_llm

class TwoStepIngestor:
    def __init__(self, workspace_dir="workspace"):
        self.raw_dir = os.path.join(workspace_dir, "raw")
        self.wiki_dir = os.path.join(workspace_dir, "wiki")
        self.converter = DocumentConverter()
        for d in [self.raw_dir, self.wiki_dir]: os.makedirs(d, exist_ok=True)
    def process_file(self, filepath):
        filename = os.path.basename(filepath)
        doc = self.converter.convert(filepath)
        text = doc.document.export_to_markdown()
        llm = get_llm()
        # Step 1: Analysis
        res1 = llm.generate([{"role":"user", "content":f"Analyze this text for entities and relationships: {text[:2000]}"}])
        # Step 2: Generation
        res2 = llm.generate([{"role":"system", "content":"Create a wiki page with YAML frontmatter."}, {"role":"user", "content":f"Text: {text[:2000]}\nAnalysis: {res1}"}])
        return {"raw_text": text, "wiki": res2}

In [ ]:
%%writefile search_engine.py
# -*- coding: utf-8 -*-
import lancedb, os
import pandas as pd
from llm_utils import get_embedder, get_reranker

class SearchEngine:
    def __init__(self, db_path="workspace/lancedb", table_name="kb"):
        os.makedirs(db_path, exist_ok=True)
        self.db = lancedb.connect(db_path)
        self.table_name = table_name
        self.embedder = get_embedder()
        self.reranker = get_reranker()
    def add_to_index(self, chunks):
        data = []
        for c in chunks:
            data.append({"vector": self.embedder.encode([c['text']])[0], "text": c['text'], "pdf_path": c['pdf_path']})
        if self.table_name in self.db.table_names():
            self.db.open_table(self.table_name).add(data)
        else:
            self.db.create_table(self.table_name, data=data)
    def search(self, query, k=5):
        if self.table_name not in self.db.table_names(): return []
        table = self.db.open_table(self.table_name)
        res = table.search(self.embedder.encode([query])[0]).limit(15).to_pandas()
        scores = self.reranker.predict([[query, t] for t in res['text']])
        res['score'] = scores
        return res.sort_values('score', ascending=False).head(k).to_dict('records')

In [ ]:
%%writefile app.py
# -*- coding: utf-8 -*-
import os, streamlit as st
from streamlit_pdf_viewer import pdf_viewer
from ingestion_engine import TwoStepIngestor
from search_engine import SearchEngine
from llm_utils import get_llm

st.set_page_config(layout="wide")
if "ingestor" not in st.session_state: st.session_state.ingestor = TwoStepIngestor()
if "search" not in st.session_state: st.session_state.search = SearchEngine()
if "messages" not in st.session_state: st.session_state.messages = []
if "v_path" not in st.session_state: st.session_state.v_path = None

with st.sidebar:
    files = st.file_uploader("Belge Yukle", accept_multiple_files=True)
    if st.button("Isle"):
        for f in files:
            p = os.path.join(st.session_state.ingestor.raw_dir, f.name)
            with open(p, "wb") as out: out.write(f.getbuffer())
            res = st.session_state.ingestor.process_file(p)
            chunks = [{"text": c, "pdf_path": p} for c in res['raw_text'].split('\n\n') if len(c) > 40]
            st.session_state.search.add_to_index(chunks)
        st.success("Hazir!")

c1, c2 = st.columns([6, 4])
with c1:
    for m in st.session_state.messages: 
        with st.chat_message(m['role']): st.markdown(m['content'])
    if q := st.chat_input("Sorunuz..."):
        st.session_state.messages.append({"role":"user", "content":q})
        with st.chat_message("user"): st.markdown(q)
        hits = st.session_state.search.search(q)
        ctx = "\n".join([f"[KAYNAK {i+1}]: {h['text']}" for i, h in enumerate(hits)])
        sys = f"Sen QwenRaggity V2'sin. Sadece baglama gore cevap ver.\n\nBAGLAM:\n{ctx}"
        with st.chat_message("assistant"):
            r = st.write_stream(get_llm().generate([{"role":"system", "content":sys}, {"role":"user", "content":q}], stream=True))
            st.session_state.messages.append({"role":"assistant", "content":r})
        if hits: 
            st.session_state.v_path = hits[0]['pdf_path']
            st.rerun()
with c2:
    if st.session_state.v_path: 
        with open(st.session_state.v_path, "rb") as f: pdf_viewer(f.read())


In [ ]:
# 2. Ollama Kurulumu ve Baslatma
import time, subprocess, re
print("🦙 Ollama kontrol ediliyor...")
os.system("curl -fsSL https://ollama.com/install.sh | sh")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)
print("🧠 Model indiriliyor...")
os.system("ollama pull qwen2.5:14b")

# 3. Streamlit ve Pinggy Baslat
print("🚀 Sistem baslatiliyor...")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"])
time.sleep(5)
os.system("ssh -p 443 -R0:localhost:8501 -o StrictHostKeyChecking=no -o ServerAliveInterval=30 a.pinggy.io > pinggy_log.txt 2>&1 &")
time.sleep(5)

if os.path.exists("pinggy_log.txt"):
    with open("pinggy_log.txt", "r") as f:
        urls = re.findall(r"https://[a-zA-Z0-9-]+\.a\.free\.pinggy\.link", f.read())
        if urls: 
            print("\n" + "="*50)
            print(f"👉 GIRIS LINKI: {urls[0]}")
            print("="*50)